# Loading, Inspecting, Cleaning, and Transforming Data

This notebook walks through a  data cleaning workflow in Python using `pandas`.


The dataset (`datania_households_raw.csv`) contains survey records from fictional households.
It has intentional data quality problems that we will discover and fix together.

## 1. Setup

### Folder structure assumed by this notebook

This notebook is designed to be saved at:

```
py4stat/
├── notebooks/
│   └── examples/
│       └── example_inspect_and_apply.ipynb   ← you are here
└── data/
    └── 0_raw/
        └── datania_households_raw.csv
```

### Relative paths

A **relative path** describes a location *relative to where you currently are*, rather than from the root of the filesystem.

Think of it like giving directions: instead of saying "go to 12 Main Street, Rome, Italy", you say "go two blocks north from here". Python does the same thing with `../`:

| Notation | Meaning |
|---|---|
| `./` | the current folder (where this notebook lives) |
| `../` | one folder up |
| `../../` | two folders up |

In this notebook the file is at `notebooks/examples/`, so to reach `data/0_raw/` we need to go **two folders up** (out of `examples/`, then out of `notebooks/`) and then back down into `data/0_raw/`:

```
notebooks/examples/  →  ../../  →  py4stat/  →  data/0_raw/
```

That is why the path starts with `'../../data/0_raw'`.

> **`os.path.join`** stitches path pieces together with the correct separator for your operating system (`/` on macOS/Linux, `\` on Windows), so your code works everywhere.


In [ ]:
import os
import pandas as pd
import numpy as np

# Path to the raw data folder, relative to this notebook's location.
# This assumes the notebook is at:  py4stat/notebooks/examples/
# and the data is at:               py4stat/data/0_raw/
#
# ../../  moves two levels up (out of 'examples/', then out of 'notebooks/')
# landing at the py4stat/ root, from where we navigate into data/0_raw/.
DATA_RAW_DIR = '../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

print('Data path:', raw_path)
print('File exists?', os.path.exists(raw_path))

# Show floats as plain numbers (no scientific notation)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Load the CSV and force identifier-like columns to be read as strings.
#
# Why? Pandas would read '01' as the integer 1, silently dropping the leading zero.
# Declaring dtype='str' for those columns prevents that from the start.
df = pd.read_csv(
    raw_path,
    dtype={'hh_id': 'str', 'region_code': 'str'}
)

# Confirm the dtypes were applied correctly
print(df[['hh_id', 'region_code']].head(8))
df.dtypes

In [ ]:
# Always start with a visual preview — get a feel for the data before doing anything else
df.head(10)

## 2. Diagnosing Structure and Data Quality

Before cleaning anything, build an overview of what you have.
Use `info()`, `isna()`, and `describe()` together — each reveals different kinds of problems.

In [ ]:
# info() shows column names, non-null counts, and inferred data types.
# Look for: unexpected 'object' types, and columns with fewer non-null values than total rows.
df.info()

In [ ]:
# Count missing values per column
# A non-zero count here means we have work to do for that column
df.isna().sum()

In [ ]:
df.describe(include="all")

**What to look for after running the cells above:**

- `income_dkw` is type `object` — why? What does that suggest about its values?
- `hh_size` has a min of `-1` and a max of `99` — are those valid household sizes?
- `age` has a max of `999` — what does that suggest?
- `survey_date` is loaded as `object` instead of a date — can you see why from the raw values?
- Are there any duplicate rows?

## 3. Inspecting Categorical Columns

Use `value_counts(dropna=False)` to count how often each value appears — including `NaN`.
This is the fastest way to spot typos, inconsistent coding, and sentinel values.

In [ ]:
# Inspect settlement type — do you spot anything unexpected?
df['urban_rural'].value_counts(dropna=False)

In [ ]:
# Inspect region codes — look for inconsistent formatting and coded missing values
df['region_code'].value_counts(dropna=False)

## 4. Cleaning Categorical Columns

From the inspection above, we found two problems:

**`urban_rural`:**
- `'Urbn'` is a typo for `'Urban'` — fix with `.replace()`

**`region_code`:**
- `'99'` is a coded missing value (not a real region) — replace with `NaN`
- `'1'` should be `'01'` to match the standard 2-digit format — fix with an exact-match dictionary

> **Tip:** Always call `value_counts()` again after each fix to verify the result.

> **Modifying in place vs. creating a new column:** Here we overwrite the original column because the fix is unambiguous (a clear typo). For more complex transformations, prefer creating a new column (e.g., `urban_rural_clean`) so you can always compare before and after.

In [ ]:
# Fix the typo: 'Urbn' → 'Urban'
df['urban_rural'] = df['urban_rural'].replace('Urbn', 'Urban')

# Verify: only 'Urban' and 'Rural' should remain
df['urban_rural'].value_counts(dropna=False)

In [ ]:
# Replace coded missing values with actual NaN
# '99' is a sentinel/placeholder used to signal "no data", not a real region code
coded_missing = ['99']
df['region_code'] = df['region_code'].replace(coded_missing, np.nan)

df['region_code'].value_counts(dropna=False)

In [ ]:
# We need to normalize '1' → '01'.
#
# WRONG approach: .str.replace('1', '01') replaces every '1' character in the string,
# so '01' becomes '001', '21' becomes '021', etc.

In [ ]:
# Correct fix: .replace() with a dictionary does exact-value matching.
# Only '1' is replaced; NaN stays NaN; all other codes are untouched.
code_map = {'1': '01'}
df['region_code'] = df['region_code'].replace(code_map)

df['region_code'].value_counts(dropna=False)

## 5. Cleaning `income_dkw` — From Messy Text to a Numeric Column

This is the main cleaning challenge. The `income_dkw` column holds household income,
but the raw values are full of problems:

- Currency prefixes and symbols: `Ar`, `$`
- Thousands separators: spaces, commas
- Coded missing values: `unknown`, `999999`, `not recorded`
- Negative values (impossible for income)

We will clean this column **step by step**, progressively building toward a robust, reusable function.

In [ ]:
# Always inspect all unique values before writing any cleaning code.
# This tells you exactly what patterns you need to handle.
df['income_dkw'].unique()

In [ ]:
# --- Step 1: Explore the cleaning pipeline interactively ---
# Chain .str operations one at a time so you can see what each step does.
#
# Key distinction:
#   .str.replace()  → operates character by character within each string
#   .replace()      → matches and replaces whole values (exact match)

income_text = df['income_dkw'].astype('string')

income_text = income_text.str.replace(' ', '', regex=False)    # remove spaces
income_text = income_text.str.replace('Ar', '', regex=False)   # remove currency prefix
income_text = income_text.str.replace(',', '', regex=False)    # remove thousands separator

income_text = income_text.replace('unknown', np.nan)           # whole-value match for missing
income_text = income_text.replace('999999', np.nan)

income_text.unique()

In [ ]:
# --- Step 2: Collapse into a single chained expression ---
# Once each step is verified, chain them together and store the result
# as a NEW column. Keep the original 'income_dkw' intact for comparison.

df['income_dkw_clean'] = (
    df['income_dkw']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.replace('Ar', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('unknown', np.nan)
    .replace('999999', np.nan)
)

# Compare original vs cleaned side by side
df[['income_dkw', 'income_dkw_clean']].head(10)

In [ ]:
# --- Step 3: Wrap the logic in a function (version 1) ---
# Functions make cleaning logic reusable and testable.
# This first version is intentionally simple — it handles the most common cases.

def clean_income_v1(value):
    """Remove formatting from a raw income value and return a cleaned string.

    Converts known invalid codes and NaN to np.nan.
    """
    text = str(value)                           # handles NaN → 'nan', int → '1200', etc.
    text = text.replace(' ', '')                # remove spaces
    text = text.replace('Ar', '')               # remove currency prefix
    text = text.replace(',', '')                # remove thousands separator
    if text in ['unknown', '99', 'nan']:        # treat these as missing
        return np.nan
    return text

# Test the function on individual values BEFORE applying to the whole column
print(clean_income_v1("Ar 32,000"))    # → '32000'
print(clean_income_v1("45 000"))       # → '45000'
print(clean_income_v1("unknown"))      # → nan
print(clean_income_v1(np.nan))         # → nan

In [ ]:
# Apply the function to the entire column.
# .apply() calls clean_income_v1 once for every value in the Series.
df['income_dkw'].apply(clean_income_v1)

In [ ]:
# --- Step 4: Make the function flexible with parameters (version 2) ---
# Hardcoded values limit reusability. By adding parameters with defaults,
# the function works out of the box but can be customized for any dataset.

def clean_income(value, chars_to_remove=None, null_codes=None):
    """Clean a messy income string by removing formatting and replacing invalid codes.

    Parameters
    ----------
    value : any
        The raw income value (string, float, NaN, int, …).
    chars_to_remove : list of str, optional
        Substrings to strip out. Defaults to common currency and formatting characters.
    null_codes : list of str, optional
        Exact values to treat as missing (converted to np.nan).

    Returns
    -------
    str or float
        Cleaned string, or np.nan if the value matched a null code.
    """
    if chars_to_remove is None:
        chars_to_remove = [' ', 'Ar', ',', '$', '%']
    if null_codes is None:
        null_codes = ['99', '999', 'unknown', 'nan', 'UNKNOWN', '999999', 'not recorded']

    text = str(value)
    for char in chars_to_remove:
        text = text.replace(char, '')

    if text in null_codes:
        return np.nan
    return text

# Quick tests
print(clean_income("Ar 32,000"))              # → '32000'  (default params)
print(clean_income("$ 1,200,000"))            # → '1200000'
print(clean_income("unknown"))                # → nan
print(clean_income("not recorded"))           # → nan
print(clean_income("--------Ar 32,0,00"))     # not fully cleaned — '-' not in default list

In [ ]:
# --- Optional: Externalise configuration to a file ---
# In real projects, the list of characters to remove is often stored in a config file
# so that analysts can update it without touching the code.

# Create and save the config file:
invalid_chars_data = pd.DataFrame({
    'invalid_chars': [' ', 'Ar', ',', '$', '-', '%', 'not recorded']
})
invalid_chars_path = os.path.join(DATA_RAW_DIR, 'invalid_chars.csv')
invalid_chars_data.to_csv(invalid_chars_path, index=False)
print(f"Saved config to: {invalid_chars_path}")

# Read it back and use it:
invalid_chars_df = pd.read_csv(invalid_chars_path)
chars_from_file = invalid_chars_df['invalid_chars'].tolist()
print("Characters loaded from file:", chars_from_file)

In [ ]:
# Test v2 with custom parameters (using chars loaded from the config file)
test_values = ["Ar 32,000", "$ 1,200,000", "--------Ar 32,0,00", "unknown", "not recorded", np.nan]

for val in test_values:
    result = clean_income(val, chars_to_remove=chars_from_file)
    print(f"  Input: {repr(str(val)):35s}  →  Output: {repr(result)}")

In [ ]:
# Apply v2 with default parameters — works exactly like v1 for standard cases
df['income_dkw'].apply(clean_income)

In [ ]:
# --- Step 5: Using a lambda to pass custom arguments to .apply() ---
#
# .apply() calls the function with one argument: the cell value.
# If your function needs extra arguments, wrap it in a lambda:
#
#   lambda x: my_function(x, param1=value1, param2=value2)
#
# The lambda receives x (the cell value) and passes it to clean_income
# along with whatever custom arguments you specify.

custom_chars = chars_from_file                                           # from config file
custom_nulls = ['99', '999', 'unknown', 'nan', 'UNKNOWN', '999999', 'not recorded']

df['income_dkw_clean'] = df['income_dkw'].apply(
    lambda x: clean_income(x, chars_to_remove=custom_chars, null_codes=custom_nulls)
)

df[['income_dkw', 'income_dkw_clean']].head(15)

### Step 6: Convert to Numeric

After cleaning the text, the column still contains strings.
Use `pd.to_numeric()` to convert to a proper number.

The `errors='coerce'` argument is key: any value that cannot be parsed as a number (e.g., remaining garbage text) is silently converted to `NaN` instead of raising an error.

In [ ]:
# Convert the cleaned text column to float.
# errors='coerce' turns un-parseable values into NaN instead of raising an error.
df['income_dkw_num'] = pd.to_numeric(df['income_dkw_clean'], errors='coerce')

# Compare all three versions side by side
df[['income_dkw', 'income_dkw_clean', 'income_dkw_num']].head(15)

## 6. Applying Functions Row-Wise with `apply(axis=1)`

Until now, `.apply()` called our function **once per cell** on a single column.

With `apply(axis=1)`, the function receives an **entire row** (as a `Series`) at each call.
This lets you combine values from multiple columns in a single operation.

```python
df.apply(my_function, axis=1)   # my_function(row) receives a full row
```

**Common use cases:**
- Deriving a column that depends on two or more other columns
- Flagging rows based on combinations of conditions
- Complex feature engineering that goes beyond simple arithmetic

In [ ]:
# --- Row-wise apply: compute income per capita ---
# We want income / hh_size, but we need to guard against edge cases:
#   - missing income         → return NaN
#   - hh_size of 0 or less  → division would be meaningless → return NaN
#
# Inside the function, access any column with row['column_name'].

def compute_income_per_capita(row):
    """Return income per household member, or NaN if inputs are invalid."""
    income  = row['income_dkw_num']
    hh_size = row['hh_size']

    if pd.isna(income):
        return np.nan
    if pd.isna(hh_size) or hh_size <= 0:
        return np.nan
    return round(income / hh_size, 2)


# axis=1 → call the function once per ROW
df['income_per_capita'] = df.apply(compute_income_per_capita, axis=1)

df[['hh_id', 'hh_size', 'income_dkw_num', 'income_per_capita']].head(15)

Let's break this apart piece by piece:

| Part | What it does |
|---|---|
| `lambda row:` | Declares an anonymous function. `row` is the name given to the current row (a pandas `Series`). |
| `row['income_dkw_num']` | Reads the value in the `income_dkw_num` column for this row. |
| `pd.notna(row['income_dkw_num'])` | Returns `True` if the income value is **not** missing. We check this first to avoid comparing `NaN > 500_000`, which would return `False` silently but is logically wrong. |
| `... and row['income_dkw_num'] > 500_000` | Only evaluated if income is not missing. Returns `True` if income exceeds 500,000. Together, this is **condition A**: *income is present and implausibly high*. |
| `row['hh_size'] <= 0` | Returns `True` if household size is zero or negative — **condition B**: *impossible household size*. |
| `(condition A) or (condition B)` | The whole expression is `True` if **at least one** condition holds. This is what gets stored in the `suspicious` column. |
| `axis=1` | Tells pandas to call the lambda **once per row**, passing the entire row as `row`. Without this, it would iterate over columns instead. |

In [ ]:
# Flag suspicious rows using a row-wise lambda.
#
# Condition A: income is present AND is implausibly high (> 500,000)
#   - pd.notna() guards against comparing NaN with >, which would silently return False
#   - 'and' short-circuits: the > check only runs if notna() is True
#
# Condition B: household size is zero or negative (impossible in reality)
#
# The row is flagged (True) if condition A OR condition B holds.
# axis=1 ensures the lambda receives one full row at a time, not one column at a time.

df['suspicious'] = df.apply(
    lambda row: (
        pd.notna(row['income_dkw_num']) and row['income_dkw_num'] > 500_000  # condition A
    ) or row['hh_size'] <= 0,                                                 # condition B
    axis=1
)

print(f"{df['suspicious'].sum()} suspicious rows found:\n")
df[df['suspicious']][['hh_id', 'hh_size', 'income_dkw', 'income_dkw_num', 'urban_rural']]

## Summary

| Concept | Tool | Example |
|---|---|---|
| Load with correct types | `pd.read_csv(dtype=…)` | `dtype={'hh_id': 'str'}` |
| Inspect structure | `.info()`, `.isna().sum()`, `.describe()` | — |
| Inspect categories | `.value_counts(dropna=False)` | — |
| Fix typos / recode | `.replace({'old': 'new'})` | `'Urbn' → 'Urban'` |
| Normalise codes | `.replace(dict)` | `{'1': '01'}` |
| Clean text column | `.str.replace()` chained | strip `Ar`, `,`, spaces |
| Reusable function | `def f(x): …` + `.apply(f)` | `clean_income_v1` |
| Flexible function | add parameters with defaults | `clean_income` v2 |
| Custom params in apply | `lambda x: f(x, param=val)` | `apply(lambda x: clean_income(x, …))` |
| Text → number | `pd.to_numeric(errors='coerce')` | `income_dkw_num` |
| Row-wise named function | `apply(func, axis=1)` | `compute_income_per_capita` |
| Row-wise lambda | `apply(lambda row: …, axis=1)` | flag suspicious records |